Calculate derivative manually

This section demonstrates how to calculate the derivative of a simple function manually. The function is f(x) = x^2, and its derivative f'(x) = 2x.

In [ ]:
def dy_dx(x):
  return 2*x

In [ ]:
dy_dx(3)

6

This section introduces PyTorch's Autograd engine, which automatically computes gradients for all operations on tensors. It's crucial for training neural networks.

Calculate derivative with Autograd

In [ ]:
import torch

In [ ]:
x = torch.tensor(3.0, requires_grad=True)

Here, we create a tensor `x` with a value of 3.0. The key argument `requires_grad=True` tells PyTorch to track all operations on this tensor so that gradients can be computed later. This tensor `x` is considered a **leaf tensor** in the computation graph because it's created by the user.

In [ ]:
y = x**2

We define `y` as a function of `x` (`y = x^2`). Since `x` has `requires_grad=True`, `y` will also have its operations tracked. `y` is a **non-leaf tensor** as it's a result of an operation.

In [ ]:
x

tensor(3., requires_grad=True)

Displaying `x` shows its value and `requires_grad=True`.

In [ ]:
y

tensor(9., grad_fn=<PowBackward0>)

Displaying `y` shows its value and `grad_fn=<PowBackward0>`. `grad_fn` indicates the function that created this tensor (in this case, `PowBackward0` for the power operation), which is used by Autograd to compute gradients during the backward pass.

In [ ]:
y.backward()

Calling `y.backward()` computes the gradients of `y` with respect to all tensors that `y` depends on and have `requires_grad=True`. In this case, it computes `dy/dx` and stores it in `x.grad`.

In [ ]:
x.grad

tensor(6.)

Accessing `x.grad` gives us the computed gradient. Since `y = x^2`, `dy/dx = 2x`. For `x=3`, `dy/dx = 2*3 = 6`. This matches the output.

In [ ]:
# another example
import math

def dz_dx(x):
    return 2 * x * math.cos(x**2)

Here's another example with a more complex function: `z = sin(x^2)`. The manual derivative `dz/dx` is calculated using the chain rule: `dz/dx = dz/dy * dy/dx = cos(y) * 2x = cos(x^2) * 2x`.

In [ ]:
dz_dx(4)

-7.661275842587077

Calculating the manual derivative for `x=4`.

In [ ]:
x = torch.tensor(4.0, requires_grad=True)

Setting up the tensor `x` for Autograd, with `requires_grad=True`.

In [ ]:
y = x ** 2

Intermediate variable `y = x^2`.

In [ ]:
z = torch.sin(y)

Final variable `z = sin(y)`.

In [ ]:
x

tensor(4., requires_grad=True)

Displaying `x`.

In [ ]:
y

tensor(16., grad_fn=<PowBackward0>)

Displaying `y`. Note `grad_fn=<PowBackward0>`.

In [ ]:
z

tensor(-0.2879, grad_fn=<SinBackward0>)

Displaying `z`. Note `grad_fn=<SinBackward0>`.

In [ ]:
z.backward()

Calling `z.backward()` to compute gradients of `z` with respect to `x`.

In [ ]:
x.grad

tensor(-7.6613)

The computed gradient `x.grad` matches the manual calculation.

In [ ]:
y.grad

/tmp/ipykernel_1794/486760323.py:1: UserWarning: The .grad attribute of a Tensor that is not a leaf Tensor is being accessed. Its .grad attribute won't be populated during autograd.backward(). If you indeed want the .grad field to be populated for a non-leaf Tensor, use .retain_grad() on the non-leaf Tensor. If you access the non-leaf Tensor by mistake, make sure you access the leaf Tensor instead. See github.com/pytorch/pytorch/pull/30531 for more information. (Triggered internally at /pytorch/build/aten/src/ATen/core/TensorBody.h:494.)
  y.grad


Attempting to access `y.grad` results in a warning. This is because `y` is a **non-leaf tensor** that was created as a result of an operation (`x**2`). By default, gradients are only accumulated for leaf tensors. To get gradients for non-leaf tensors, you would need to call `y.retain_grad()` before the backward pass.

A perceptron (calculate with autograd)

This section demonstrates how to use Autograd for a simple perceptron, including calculating gradients both manually and automatically. A perceptron is a basic unit of a neural network.

In [ ]:
import torch

# Inputs
x = torch.tensor(6.7)  # Input feature
y = torch.tensor(0.0)  # True label (binary)

w = torch.tensor(1.0)  # Weight
b = torch.tensor(0.0)  # Bias

We define the inputs `x` (feature), `y` (true label), `w` (weight), and `b` (bias). Note that initially, `w` and `b` do not have `requires_grad=True` for the manual calculation part.

In [ ]:
# Binary Cross-Entropy Loss for scalar
def binary_cross_entropy_loss(prediction, target):
    epsilon = 1e-8  # To prevent log(0)
    prediction = torch.clamp(prediction, epsilon, 1 - epsilon)
    return -(target * torch.log(prediction) + (1 - target) * torch.log(1 - prediction))

This helper function calculates the Binary Cross-Entropy Loss, a common loss function for binary classification tasks. `epsilon` is added to prevent `log(0)` errors.

In [ ]:
# Forward pass
z = w * x + b  # Weighted sum (linear part)
y_pred = torch.sigmoid(z)  # Predicted probability

# Compute binary cross-entropy loss
loss = binary_cross_entropy_loss(y_pred, y)

This is the **forward pass** of the perceptron:
1.  `z = w * x + b`: Calculates the weighted sum of inputs and bias.
2.  `y_pred = torch.sigmoid(z)`: Applies the sigmoid activation function to `z` to get a predicted probability between 0 and 1.
Then, the `loss` is computed using the `binary_cross_entropy_loss` function.

In [ ]:
loss

tensor(6.7012)

Displaying the calculated loss.

In [ ]:
# Calculate Derivatives Manually:
# 1. dL/d(y_pred): Loss with respect to the prediction (y_pred)
dloss_dy_pred = (y_pred - y)/(y_pred*(1-y_pred))

# 2. dy_pred/dz: Prediction (y_pred) with respect to z (sigmoid derivative)
dy_pred_dz = y_pred * (1 - y_pred)

# 3. dz/dw and dz/db: z with respect to w and b
dz_dw = x  # dz/dw = x
dz_db = 1  # dz/db = 1 (bias contributes directly to z)

dL_dw = dloss_dy_pred * dy_pred_dz * dz_dw
dL_db = dloss_dy_pred * dy_pred_dz * dz_db

Here, the gradients are calculated **manually** using the chain rule for backpropagation:
*   **dL/d(y_pred)**: Derivative of the loss with respect to the predicted probability.
*   **dy_pred/dz**: Derivative of the sigmoid function `y_pred = sigmoid(z)`.
*   **dz/dw** and **dz/db**: Derivatives of `z = w*x + b` with respect to `w` and `b`.
Finally, `dL_dw` and `dL_db` are computed by multiplying these partial derivatives according to the chain rule.

In [ ]:
print(f"Manual Gradient of loss w.r.t weight (dw): {dL_dw}")
print(f"Manual Gradient of loss w.r.t bias (db): {dL_db}")

Manual Gradient of loss w.r.t weight (dw): 6.691762447357178
Manual Gradient of loss w.r.t bias (db): 0.998770534992218


Printing the manually calculated gradients.

In [ ]:
x = torch.tensor(6.7)
y = torch.tensor(0.0)

Re-initializing `x` and `y`.

In [ ]:
w = torch.tensor(1.0, requires_grad=True)
b = torch.tensor(0.0, requires_grad=True)

Now, we re-initialize `w` and `b`, but this time with `requires_grad=True`, so Autograd can track operations and compute their gradients automatically.

In [ ]:
w

tensor(1., requires_grad=True)

Displaying `w` with `requires_grad=True`.

In [ ]:
b

tensor(0., requires_grad=True)

Displaying `b` with `requires_grad=True`.

In [ ]:
z = w*x + b
z

tensor(6.7000, grad_fn=<AddBackward0>)

Performing the forward pass for `z`. Notice `grad_fn=<AddBackward0>` for `z`.

In [ ]:
y_pred = torch.sigmoid(z)
y_pred

tensor(0.9988, grad_fn=<SigmoidBackward0>)

Calculating `y_pred` using `sigmoid`. Notice `grad_fn=<SigmoidBackward0>` for `y_pred`.

In [ ]:
loss = binary_cross_entropy_loss(y_pred, y)
loss

tensor(6.7012, grad_fn=<NegBackward0>)

Calculating the `loss`. Notice `grad_fn=<NegBackward0>` for `loss`.

In [ ]:
loss.backward()

Calling `loss.backward()` triggers the automatic gradient computation. PyTorch traverses the computation graph backward from `loss` to `w` and `b` (the leaf tensors with `requires_grad=True`), computing and accumulating gradients.

In [ ]:
print(w.grad)
print(b.grad)

tensor(6.6918)
tensor(0.9988)


Printing `w.grad` and `b.grad` shows the gradients computed by Autograd, which should match the manually calculated values. This demonstrates the power and convenience of Autograd.

## Autograd with 2D tensor as input

This section illustrates Autograd's behavior with a 2D tensor (a vector) as input, specifically how gradients are handled when operations like `mean()` are applied.

In [ ]:
x = torch.tensor([1.0, 2.0, 3.0], requires_grad=True)

We create a 1D tensor `x` with multiple elements, again setting `requires_grad=True`.

In [ ]:
x

tensor([1., 2., 3.], requires_grad=True)

Displaying the tensor `x`.

In [ ]:
y = (x**2).mean()
y

tensor(4.6667, grad_fn=<MeanBackward0>)

Here, `y` is calculated by squaring each element of `x` and then taking the mean of the resulting squared values. So, if `x = [x1, x2, x3]`, then `y = (x1^2 + x2^2 + x3^2) / 3`. The `grad_fn` is `MeanBackward0`.

In [ ]:
y.backward()

Calling `y.backward()` computes the gradients of `y` with respect to each element of `x`. The derivative of `y` with respect to `xi` is `(2 * xi) / 3`.
For `x=[1, 2, 3]`:
*   `x1`: (2 * 1) / 3 = 0.6667
*   `x2`: (2 * 2) / 3 = 1.3333
*   `x3`: (2 * 3) / 3 = 2.0000

In [ ]:
x.grad

tensor([0.6667, 1.3333, 2.0000])

The `x.grad` tensor correctly shows these computed gradients.

In [ ]:
# clearing grad
x = torch.tensor(2.0, requires_grad=True)
x

tensor(2., requires_grad=True)

When `backward()` is called multiple times on the same graph, gradients are **accumulated** by default. To prevent this, or to reset gradients for a new backward pass, you must explicitly clear them. This section demonstrates how to do that.

In [ ]:
y = x ** 2
y

tensor(4., grad_fn=<PowBackward0>)

We recalculate `y = x^2` for `x=2.0`.

In [ ]:
y.backward()

First `backward()` call. `x.grad` will be `2 * x = 4`.

In [ ]:
x.grad

tensor(4.)

Displaying `x.grad` after the first backward pass.

In [ ]:
x.grad.zero_()

tensor(0.)

Calling `x.grad.zero_()` explicitly sets the gradient of `x` to zero. This is crucial before performing another backward pass in an iterative training loop (e.g., in neural networks) to ensure gradients from previous iterations don't interfere.

In [ ]:
# disable gradient tracking
x = torch.tensor(2.0, requires_grad=True)
x

tensor(2., requires_grad=True)

Sometimes you need to perform operations on tensors without tracking gradients. This is useful for inference, or when you want to freeze certain parts of a model during training. This section shows different ways to disable gradient tracking.

In [ ]:
y = x ** 2
y

tensor(4., grad_fn=<PowBackward0>)

First, we re-initialize `x` with `requires_grad=True`.

In [ ]:
y.backward()

Calculating `y = x**2` again.

In [ ]:
x.grad

tensor(8.)

After `y.backward()`, `x.grad` is `2*4 = 8` (since the previous `x.grad` was 4 and it accumulated). Let's re-run this block to get clean state or explicitly zero the grad.

In [ ]:
# option 1 - requires_grad_(False)
# option 2 - detach()
# option 3 - torch.no_grad()

This is a summary of the three main ways to disable gradient tracking:
1.  `tensor.requires_grad_(False)`: Modifies the `requires_grad` attribute of the tensor in-place.
2.  `tensor.detach()`: Creates a new tensor that shares the same data but does not participate in the gradient computation.
3.  `with torch.no_grad():`: A context manager that disables gradient calculation for any operations within its block. This is often preferred for inference as it's cleaner and safer.

In [ ]:
x.requires_grad_(False)

tensor(2.)

Using `x.requires_grad_(False)` directly changes the `requires_grad` property of the tensor `x` to `False` in-place. This means `x` will no longer track gradients for future operations.

In [ ]:
x

tensor(2.)

Now `x` is displayed without `requires_grad=True`.

In [ ]:
y = x ** 2

When `y = x ** 2` is calculated after `x.requires_grad_(False)`, `y` will also not track gradients because its input `x` doesn't.

In [ ]:
y

tensor(4.)

Displaying `y`. Notice there is no `grad_fn` as gradients are not being tracked.

In [ ]:
y.backward()

RuntimeError: element 0 of tensors does not require grad and does not have a grad_fn

Attempting to call `y.backward()` now results in a `RuntimeError` because `y` does not have a `grad_fn` and does not require gradients, so there's no computation graph to backpropagate through.

In [ ]:
x = torch.tensor(2.0, requires_grad=True)
x

tensor(2., requires_grad=True)

Here we demonstrate `detach()`. We re-initialize `x` with `requires_grad=True`.

In [ ]:
z = x.detach()
z

tensor(2.)

`z = x.detach()` creates a new tensor `z` that is a view of `x` but has `requires_grad=False`. It effectively 'detaches' `z` from the computation graph that `x` is part of. Operations on `z` will not track gradients back to `x`.

In [ ]:
y = x ** 2

`y = x ** 2` will still track gradients because `x` still has `requires_grad=True`.

In [ ]:
y

tensor(4., grad_fn=<PowBackward0>)

Displaying `y`. It correctly shows `grad_fn=<PowBackward0>`.

In [ ]:
y1 = z ** 2
y1

tensor(4.)

`y1 = z ** 2` creates `y1`. Since `z` was detached, `y1` also does not track gradients.

In [ ]:
y.backward()

Calling `y.backward()` successfully computes gradients for `x` because `y` was part of the original computation graph.

In [ ]:
y1.backward()

RuntimeError: element 0 of tensors does not require grad and does not have a grad_fn

Attempting `y1.backward()` results in a `RuntimeError` because `y1` does not track gradients (due to `z` being detached), similar to the `requires_grad_(False)` case.

In [ ]:
x = torch.tensor(2.0, requires_grad=True)
x

tensor(2., requires_grad=True)

Finally, we set up `x` again.

In [ ]:
y = x ** 2

And `y`.

In [ ]:
y

tensor(4., grad_fn=<PowBackward0>)

Displaying `y`.

In [ ]:
y.backward()

And compute `y.backward()` one last time to ensure `x.grad` is populated correctly for the next code block (if any, though it seems to be the end of the provided code).